# Semana 1 — Poner la máquina en marcha**Proyecto:** Detección automática de fracturas óseas — ETRR**Notebook:** `01_exploracion.ipynb`---### Qué hacemos hoyNada de inteligencia artificial todavía. Hoy conseguimos que la máquina ande ymiramos los datos con nuestros propios ojos. Ese orden no es capricho: casitodos los proyectos que fracasan lo hacen porque nadie miró las imágenes.1. Verificar que tenemos GPU2. Traer el código del repositorio3. Descargar FracAtlas (4.083 radiografías)4. Armar una tabla con una fila por imagen5. Mirar 20 radiografías y sus histogramas6. Guardar la ficha del dataset en la bitácora### Criterio de cierre de la semana- [ ] Este notebook corre de principio a fin sin errores, en Colab **y** en la PC con GPU- [ ] `torch.cuda.is_available()` devuelve `True` en la PC local- [ ] El repositorio está en GitHub con su README- [ ] `docs/ficha-dataset.md` existe y tiene los números del dataset> ⚠️ **Descargo:** herramienta educativa. No es un dispositivo médico y no debe> usarse para decisiones clínicas.

---## 1. ¿Con qué máquina nos tocó?Colab te asigna una máquina virtual al azar. A veces con GPU, a veces sin ella.Conviene saberlo antes de esperar media hora a que algo entrene.**Para pedir GPU en Colab:** menú `Entorno de ejecución` → `Cambiar tipo deentorno de ejecución` → `Acelerador por hardware: GPU` → Guardar.`nvidia-smi` es el programa que le pregunta a la placa cómo está. El signo `!`adelante significa "esto no es Python, es un comando de la terminal".

In [ ]:
!nvidia-smi || echo "Sin GPU asignada. Andá a Entorno de ejecución > Cambiar tipo de entorno."

Ahora la misma pregunta, pero desde Python. Esta es la línea que aparece en elcriterio de cierre de la semana: si en la PC de la escuela devuelve `False`,falta instalar los drivers o la versión de PyTorch con soporte CUDA.

In [ ]:
try:    import torch    print(f"PyTorch          : {torch.__version__}")    print(f"¿GPU disponible? : {torch.cuda.is_available()}")    if torch.cuda.is_available():        print(f"Placa            : {torch.cuda.get_device_name(0)}")        memoria = torch.cuda.get_device_properties(0).total_memory / 1e9        print(f"Memoria          : {memoria:.1f} GB")    else:        print("\nSin GPU: hoy no importa (solo miramos imágenes).")        print("A partir de la semana 3 sí importa, y mucho.")except ImportError:    print("PyTorch no está instalado. En Colab ya viene; en la PC local:")    print("   pip install torch torchvision")

---## 2. Traer nuestro códigoAcá hay una decisión de fondo del proyecto: **el repositorio es la fuente deverdad, no el notebook.**Las funciones para descargar datos y dibujar gráficos viven en la carpeta`src/` del repositorio. El notebook solo las llama. Así, cuando arreglemos unerror, lo arreglamos en un lugar y no en los ocho notebooks donde estabacopiado.En Colab, "traer el código" significa clonar el repositorio desde GitHub.En la PC local no hace falta: el notebook ya está adentro del repo.

In [ ]:
import sys, osfrom pathlib import PathREPO_URL = "https://github.com/resagri-fiuba/Deteccion-De-Fracturas-ETRR.git"CARPETA  = "Deteccion-De-Fracturas-ETRR"def en_colab():    try:        import google.colab  # noqa        return True    except ImportError:        return Falseif en_colab():    if not Path(CARPETA).exists():        !git clone -q $REPO_URL    else:        !cd $CARPETA && git pull -q          # traer los últimos cambios    RAIZ = Path("/content") / CARPETAelse:    # Corriendo local: subimos desde notebooks/ hasta la raíz del repo    RAIZ = Path.cwd()    while not (RAIZ / "src").exists() and RAIZ != RAIZ.parent:        RAIZ = RAIZ.parentsys.path.insert(0, str(RAIZ))os.chdir(RAIZ)print(f"Raíz del proyecto: {RAIZ}")print(f"¿Encuentro src/?   {(RAIZ / 'src').exists()}")

In [ ]:
# Importamos nuestros propios módulos. Si esto falla, la celda anterior no# encontró el repositorio.from src import config, datos, visualconfig.crear_carpetas()config.fijar_semilla()          # todo lo aleatorio va a salir igual siempreprint("Módulos cargados.")print(f"Los datos van a: {config.DATOS}")

---## 3. Descargar FracAtlas**FracAtlas** son 4.083 radiografías de mano, pierna, cadera y hombro, de lascuales 717 tienen fractura. Es un dataset público, anonimizado y con licenciaCC BY 4.0 — o sea, se puede usar citándolo.La descarga son unos cuantos cientos de megas. En Colab tarda un par deminutos; se hace **una sola vez** por sesión, porque la función se fija si elarchivo ya está antes de bajarlo de nuevo.> Si la descarga falla, el error te dice exactamente qué hacer: bajar el zip a> mano desde Figshare y subirlo. Los servidores académicos se caen seguido; no> es culpa del código.

In [ ]:
carpeta = datos.preparar_fracatlas()print(f"\nDataset en: {carpeta}")

---## 4. De una carpeta de archivos a una tablaEste paso parece burocrático y es el más importante del notebook.Una carpeta con miles de imágenes no sirve para trabajar. Lo que sirve es una**tabla**: una fila por radiografía, con su ruta y su etiqueta. Todo lo queviene en las próximas ocho semanas —particiones, entrenamiento, métricas— salede esta tabla. Si la tabla está mal, todo lo demás está mal y no nos vamos aenterar hasta el final.

In [ ]:
tabla = datos.tabla_imagenes()tabla.head(8)

In [ ]:
datos.resumen(tabla)

### 🔴 Pará acá y leé el númeroEl resumen dice que **un modelo que siempre respondiera "sin fractura"acertaría el 82,4% de las veces.**Ese modelo no mira la imagen. No aprende nada. Es una línea de código quedevuelve siempre lo mismo. Y sacaría 82,4% de *accuracy*.Cuando dentro de dos semanas entrenemos algo y dé 85%, la pregunta correcta nova a ser "¿está bien 85%?" sino "**¿85% comparado con qué?**". La respuesta es:comparado con 82,4%, que se consigue sin hacer nada.Esto es lo que hace que la semana 4 —la de métricas— sea la más importante delproyecto. Anotá el número: **82,4%.**

---## 5. Mirar las radiografíasAhora sí, la parte que la mayoría se saltea.Vamos a mostrar 20 imágenes al azar con su etiqueta. Mientras las mirás, buscáespecíficamente estas cosas:- ¿Hay **texto o números quemados** en la imagen? (fecha, nombre del equipo, "L"/"R")- ¿Están todas **en la misma orientación**?- ¿Alguna es tan oscura o tan clara que **no se ve el hueso**?- ¿Se nota a simple vista cuáles tienen fractura? (spoiler: casi nunca)Lo que encuentres acá anotalo en la bitácora. En la semana 7, cuando veamos losmapas de calor, vamos a volver a esta lista.

In [ ]:
muestra = tabla.sample(20, random_state=config.SEMILLA)visual.grilla(    rutas=muestra["ruta"].tolist(),    etiquetas=muestra["fractura"].tolist(),    filas=4, columnas=5,    titulo="20 radiografías al azar — rojo = fractura",    guardar=config.SALIDAS / "s01_grilla.png",);

---## 6. El histograma: la imagen como númerosUna radiografía es una **matriz de números**. Cada píxel es un valor de 0(negro) a 255 (blanco). Nada más que eso.El histograma cuenta cuántos píxeles hay de cada valor. Sirve para ver elcontraste de un vistazo: si todos los píxeles se amontonan en una franjaangosta, la placa está lavada y hay poca información para el modelo.Corré la celda varias veces cambiando el número de la primera línea y compará.

In [ ]:
i = 0     # 🔧 cambiá este número y volvé a correr la celdavisual.histograma(    muestra.iloc[i]["ruta"],    guardar=config.SALIDAS / "s01_histograma.png",);

### La misma imagen, vista como lo que realmente esPara que quede claro que no hay magia: acá abajo imprimimos un pedacito de lamatriz. Diez filas por diez columnas de la esquina superior izquierda.

In [ ]:
import numpy as npimg = visual.cargar_gris(muestra.iloc[0]["ruta"])print(f"Tipo de objeto : {type(img).__name__}")print(f"Forma (shape)  : {img.shape}   ->  (alto, ancho) en píxeles")print(f"Tipo de dato   : {img.dtype}   ->  enteros de 0 a 255")print(f"Total de píxeles: {img.size:,}")print()print("Esquina superior izquierda, 10x10:")print(img[:10, :10])

---## 7. Estadísticas de todo el datasetUna imagen sola no dice mucho. Veamos 200 y busquemos las raras: las que tienenun tamaño muy distinto, o un contraste que se sale de la manada. Esas son lasque después rompen el entrenamiento.

In [ ]:
import pandas as pdfrom tqdm.auto import tqdmsub = tabla.sample(200, random_state=config.SEMILLA)stats = pd.DataFrame([visual.describir(r) for r in tqdm(sub["ruta"], desc="Midiendo")])stats[["alto", "ancho", "media", "desvio"]].describe().round(1)

In [ ]:
# Las 5 imágenes más oscuras y las 5 más claras del subconjunto.# ¿Son legibles? Miralas antes de seguir.raras = pd.concat([stats.nsmallest(5, "media"), stats.nlargest(5, "media")])rutas_raras = [sub[sub["archivo"] == a]["ruta"].iloc[0] for a in raras["archivo"]]visual.grilla(rutas_raras, filas=2, columnas=5,              titulo="Las 5 más oscuras y las 5 más claras",              guardar=config.SALIDAS / "s01_extremos.png");

---## 8. Escribir la ficha del datasetÚltimo paso y parte del criterio de cierre. La ficha es un archivo de texto conlos números del dataset. Va al repositorio, se cita en el informe final y evitala escena clásica de noviembre: *"¿cuántas imágenes tenía esto?"*.

In [ ]:
n = len(tabla)con = int(tabla["fractura"].sum())sin = n - conficha = f'''# Ficha del dataset — FracAtlasGenerada automáticamente por `notebooks/01_exploracion.ipynb`.| | ||---|---|| Radiografías totales | {n:,} || Con fractura | {con:,} ({con/n:.1%}) || Sin fractura | {sin:,} ({sin/n:.1%}) || Accuracy del modelo trivial | **{sin/n:.1%}** |**Modelo trivial** = responder siempre "sin fractura". Es el número a superar.## Tamaños de imagen (muestra de 200)| | alto | ancho ||---|---|---|| mínimo | {stats['alto'].min()} | {stats['ancho'].min()} || mediana | {stats['alto'].median():.0f} | {stats['ancho'].median():.0f} || máximo | {stats['alto'].max()} | {stats['ancho'].max()} |## Observaciones a ojo_(completar a mano después de mirar la grilla)_- ¿Hay texto quemado en las imágenes?- ¿Orientaciones distintas?- ¿Imágenes ilegibles?## Cita obligatoriaAbedeen, I. et al. FracAtlas: A Dataset for Fracture Classification,Localization and Segmentation of Musculoskeletal Radiographs.*Scientific Data* 10, 521 (2023). Licencia CC BY 4.0.'''destino = config.DOCS / "ficha-dataset.md"destino.write_text(ficha, encoding="utf-8")print(f"Escrita: {destino}")print()print(ficha)

---## 9. Guardar el trabajo en GitHubSi estás en Colab, los archivos que acabás de generar viven en una máquinavirtual que se borra sola. Para conservarlos hay que subirlos.Lo más simple y lo que recomendamos esta semana: **descargá `ficha-dataset.md`y `salidas/s01_grilla.png`** desde el panel de archivos de la izquierda(ícono de carpeta → botón derecho → Descargar), y subilos al repositorio desdela web de GitHub, con el botón `Add file` → `Upload files`.Desde la PC local, en cambio, es la secuencia de siempre:```bashgit add docs/ficha-dataset.md salidas/s01_grilla.pnggit commit -m "Semana 1: ficha del dataset y exploración inicial"git push```> **Por qué no subimos las imágenes del dataset:** son cientos de megas y GitHub> las rechaza. El archivo `.gitignore` ya se encarga de ignorar la carpeta> `datos/`. Al repositorio va el **código que descarga los datos**, no los datos.---## 🔧 EjerciciosPara hacer **editando este notebook**. No hay forma de aprender esto leyendo.1. **Fácil.** Cambiá la grilla para que muestre 30 imágenes en vez de 20.   (Pista: dos números en la llamada a `visual.grilla`, y uno en `sample`.)2. **Fácil.** Mostrá una grilla con **solo** radiografías con fractura.   (Pista: `tabla[tabla["fractura"] == 1]`.)3. **Media.** En `src/visual.py`, la función `histograma` dibuja una línea roja   en la media. Agregale una línea azul punteada en la **mediana**.   (Pista: `np.median(img)`.)4. **Media.** ¿Las radiografías con fractura son en promedio más oscuras que   las normales? Calculá la media de intensidad de 100 de cada grupo y compará.   Si diera una diferencia grande, tendríamos un problema serio: significaría   que el modelo podría "adivinar" mirando el brillo, sin mirar el hueso.5. **Difícil.** Escribí una función que reciba la tabla y devuelva las 10   imágenes **más parecidas entre sí** por tamaño y contraste. ¿Hay duplicados   en el dataset? Si un duplicado cae en entrenamiento y otro en test, las   métricas mienten.---**Siguiente:** `02_particion.ipynb` — separar train/val/test por paciente,que es donde se arruinan la mitad de estos proyectos.Repositorio: https://github.com/resagri-fiuba/Deteccion-De-Fracturas-ETRR